In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 2.6 Rigid-Body Rotation and the Spinning Top

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume II — Analytical Mechanics",
    number="2.6",
    title="Rigid-Body Rotation and the Spinning Top",
    blurb="The inertia tensor, Euler's equations, free precession, and the "
    "tennis-racket theorem — why a spinning body is stable about two "
    "axes and tumbles periodically about the third.",
    difficulty="advanced",
    estimate="90–110 min",
)

## Notebook overview

A rigid body has no internal degrees of freedom (every distance is frozen), so all
that is left is *orientation*. That sounds simple, and yet a spinning body is one of
the richest small systems in mechanics: a thrown phone, a tumbling wing-nut, the
wobble of the Earth's axis all live here. This notebook builds the machinery and
then *shows* it move.

The organising idea is to change frames. The mass distribution is fixed in the
**body frame**, so the **inertia tensor** $\mathsf I$ is constant there, and we can
diagonalise it once: the eigenvectors are the **principal axes** and the
eigenvalues the **principal moments** $I_1,I_2,I_3$ (the same eigenvalue problem
behind normal modes in [§1.5](../01-elementary-mechanics/coupled-oscillators.ipynb)). In that frame the angular momentum is just
$\mathbf L=\mathsf I\boldsymbol\omega$ component by component, and transporting
$d\mathbf L/dt=0$ from space into the rotating body frame gives **Euler's
equations**: three coupled nonlinear ODEs for $\boldsymbol\omega(t)$.

From there the physics unfolds. Torque-free, both the kinetic energy $T$ and
$|\mathbf L|$ are conserved, so $\boldsymbol\omega$ is pinned to the intersection of
an **energy ellipsoid** and an **angular-momentum ellipsoid**: the *polhode*, a
genuinely beautiful curve. A symmetric body ($I_1=I_2$) precesses freely at a rate
we can predict and measure. And linearising Euler's equations exposes the
headline result, the **tennis-racket (Dzhanibekov) theorem**: spin is stable about
the largest and smallest principal axes but *unstable* about the intermediate one,
where the body tumbles end over end. We animate exactly that, a 3-D body spun about
each axis, and then animate a symmetric top's symmetry axis sweeping its precession
cone in space.

> **How to read the checks.** Each exercise ends with a validation that compares
> a computed result to an expected physical fact. A ✗ does *not* by itself mean
> the physics is wrong: it means the output didn't match what the check
> expected, which may be a real error, a different-but-valid convention (a sign,
> a unit, an array order), or simply too tight a tolerance. Treat a ✗ as a prompt
> to locate the discrepancy; passing is strong evidence of correctness, not proof.

> **Scope.** This is a working review, not a textbook chapter. For the inertia
> tensor, Euler's equations, and the heavy symmetric top in full, see Nolting,
> *Theoretical Physics 2* {cite}`nolting2`, Goldstein, Poole & Safko
> {cite}`goldstein` (ch. 4–5), and Landau & Lifshitz, *Mechanics*
> {cite}`landau_mechanics` (§VI).

## Theory in brief

### The inertia tensor and principal axes

The rotational analogue of mass is the **inertia tensor**, defined (about the
centre of mass) by

```{math}
:label: eq-inertia
I_{ij} = \int \rho(\mathbf r)\,\bigl(r^2\delta_{ij} - x_i x_j\bigr)\,dV ,
```

a symmetric $3\times3$ matrix. Being symmetric, it is diagonalised by an orthogonal
change of axes: its eigenvectors are the **principal axes** fixed in the body, and
its eigenvalues the **principal moments** $I_1,I_2,I_3$. Working in the principal
(body) frame makes everything that follows diagonal: the same trick as resolving a
coupled oscillator into normal modes ([§1.5](../01-elementary-mechanics/coupled-oscillators.ipynb)).

### Angular momentum and kinetic energy

In the principal frame the angular momentum and rotational kinetic energy are

```{math}
:label: eq-Trot
\mathbf L = \mathsf I\,\boldsymbol\omega = (I_1\omega_1,\,I_2\omega_2,\,I_3\omega_3),
\qquad
T = \tfrac12\,\boldsymbol\omega\cdot\mathsf I\,\boldsymbol\omega
  = \tfrac12\bigl(I_1\omega_1^2+I_2\omega_2^2+I_3\omega_3^2\bigr).
```

Note that $\mathbf L$ and $\boldsymbol\omega$ are *not* parallel unless the body
spins about a principal axis: the source of all the interesting behaviour.

### Euler's equations

In an inertial (space) frame a torque-free body conserves $\mathbf L$:
$d\mathbf L/dt|_{\text{space}}=0$. But $\mathsf I$ is only constant in the body
frame, so we transport the derivative using
$d/dt|_{\text{space}}=d/dt|_{\text{body}}+\boldsymbol\omega\times$. With
$\mathbf L=\mathsf I\boldsymbol\omega$ this gives **Euler's equations**,

```{math}
:label: eq-euler-rigid
I_1\dot\omega_1 = (I_2-I_3)\,\omega_2\omega_3, \quad
I_2\dot\omega_2 = (I_3-I_1)\,\omega_3\omega_1, \quad
I_3\dot\omega_3 = (I_1-I_2)\,\omega_1\omega_2 ,
```

three coupled nonlinear ODEs for the body-frame angular velocity. Torque-free, they
conserve both $T$ {eq}`eq-Trot` and $|\mathbf L|$, so $\boldsymbol\omega(t)$ is
confined to the intersection of the **energy ellipsoid**
$\sum_i I_i\omega_i^2=2T$ and the **angular-momentum ellipsoid**
$\sum_i I_i^2\omega_i^2=L^2$ in $\boldsymbol\omega$-space (a sphere only in
$\mathbf L$-space): the **polhode**.

### Free precession of a symmetric top

If two moments are equal, $I_1=I_2$, the third Euler equation gives
$\dot\omega_3=0$: the spin about the symmetry axis is constant. The other two then
read $\dot\omega_1=-\Omega\omega_2$, $\dot\omega_2=+\Omega\omega_1$, so the
transverse part of $\boldsymbol\omega$ rotates at the **free-precession rate**

```{math}
:label: eq-precession
\Omega = \frac{I_3-I_1}{I_1}\,\omega_3 .
```

### Stability of steady rotation

Steady spin $\boldsymbol\omega=n\,\hat{\mathbf e}_k$ about a principal axis is an
exact solution. Linearising {eq}`eq-euler-rigid` about it, the transverse
perturbations grow like $e^{\sqrt{\sigma}\,t}$ with
$\sigma=-n^2(I_k-I_i)(I_k-I_j)/(I_iI_j)$, so the motion is bounded (stable) exactly
when

```{math}
:label: eq-stability
(I_k-I_i)(I_k-I_j) > 0 ,
```

i.e. when $I_k$ is the **largest or smallest** moment. About the **intermediate**
axis the product is negative, $\sigma>0$, and the body tumbles: the
**tennis-racket / Dzhanibekov theorem**.

---
## Setup

We carry the principal moments as a tuple `I = (I1, I2, I3)` and work in the body
frame. The reusable core:

- `euler_eqs(t, w, I)`: the right-hand side of Euler's equations
  {eq}`eq-euler-rigid`.
- `T_rot(w, I)` and `L_body(w, I)`: the conserved energy and angular momentum
  {eq}`eq-Trot`.
- `integrate_body(w0, I, t_eval)`: integrates $\boldsymbol\omega(t)$ **together
  with the orientation matrix** $\mathsf R(t)$ (body $\to$ space), via the kinematic
  relation $\dot{\mathsf R}=\mathsf R\,[\boldsymbol\omega]_\times$. We need $\mathsf
  R$ to *draw* the body in space for the animations; its columns are the body's
  principal axes expressed in the lab frame.
- `box_polys(R, half)`: the six faces of a rectangular body, rotated by $\mathsf
  R$, for the 3-D renders.

The principal axes of a rigid box are sketched in {numref}`fig-rigid-axes`.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from ecp import draw, validate
from ecp.animate import show


def euler_eqs(_t, w, I):
    """Euler's equations (eq-euler-rigid): body-frame dω/dt, torque-free.

    Parameters
    ----------
    _t : float
        Time (unused).
    w : array_like
        Body angular velocity.
    I : array_like
        Principal moments of inertia.

    Returns
    -------
    numpy.ndarray
        The angular acceleration dω/dt.
    """
    I1, I2, I3 = I
    w1, w2, w3 = w
    return np.array(
        [
            (I2 - I3) * w2 * w3 / I1,
            (I3 - I1) * w3 * w1 / I2,
            (I1 - I2) * w1 * w2 / I3,
        ]
    )


def T_rot(w, I):
    """Rotational kinetic energy T = (1/2) Σ I_i ω_i^2 (eq-Trot).

    Parameters
    ----------
    w : array_like
        Body angular velocity.
    I : array_like
        Principal moments.

    Returns
    -------
    float
        The rotational kinetic energy.
    """
    I1, I2, I3 = I
    return 0.5 * (I1 * w[0] ** 2 + I2 * w[1] ** 2 + I3 * w[2] ** 2)


def L_body(w, I):
    """Angular-momentum vector in the body frame, L = I·ω (eq-Trot).

    Parameters
    ----------
    w : array_like
        Body angular velocity.
    I : array_like
        Principal moments.

    Returns
    -------
    numpy.ndarray
        The body-frame angular momentum.
    """
    I1, I2, I3 = I
    return np.array([I1 * w[0], I2 * w[1], I3 * w[2]])


def _skew(w):
    """Skew-symmetric matrix [ω]_x with [ω]_x v = ω × v.

    Parameters
    ----------
    w : array_like
        A 3-vector.

    Returns
    -------
    numpy.ndarray
        The 3x3 skew-symmetric matrix.
    """
    return np.array([[0.0, -w[2], w[1]], [w[2], 0.0, -w[0]], [-w[1], w[0], 0.0]])


def _rigid_rhs(_t, y, I):
    """Combined state derivative for ω and the orientation R (dR/dt = R·[ω]_x)."""
    w = y[:3]
    R = y[3:].reshape(3, 3)
    dw = euler_eqs(_t, w, I)
    dR = R @ _skew(w)
    return np.concatenate([dw, dR.ravel()])


def integrate_body(w0, I, t_eval, R0=None):
    """Integrate angular velocity and body-to-space orientation together.

    Advances ω(t) by Euler's equations and the rotation R(t) alongside it, so
    the body can be drawn in space.

    Parameters
    ----------
    w0 : array_like
        Initial body angular velocity.
    I : array_like
        Principal moments.
    t_eval : numpy.ndarray
        Output times.
    R0 : numpy.ndarray, optional
        Initial orientation (default identity).

    Returns
    -------
    tuple
        The angular-velocity history and the orientation history.
    """
    if R0 is None:
        R0 = np.eye(3)
    y0 = np.concatenate([np.asarray(w0, float), R0.ravel()])
    sol = solve_ivp(
        _rigid_rhs,
        (t_eval[0], t_eval[-1]),
        y0,
        args=(I,),
        method="DOP853",
        t_eval=t_eval,
        rtol=1e-11,
        atol=1e-13,
    )
    w = sol.y[:3]
    R = sol.y[3:].T.reshape(-1, 3, 3)
    return w, R


# Rectangular body for the 3-D renders, drawn from its half-extents.
_BOX_FACES = [
    [0, 1, 2, 3],
    [4, 5, 6, 7],
    [0, 1, 5, 4],
    [2, 3, 7, 6],
    [1, 2, 6, 5],
    [0, 3, 7, 4],
]
FACE_COLORS = ["#c0851a", "#e3b667", "#46506b", "#7b86a6", "#16213e", "#3a4a6a"]


def box_polys(R, half):
    """The six face polygons of a box rotated by an orientation matrix.

    For rendering the spinning body's faces in 3-D.

    Parameters
    ----------
    R : numpy.ndarray
        Orientation matrix.
    half : array_like
        Box half-extents.

    Returns
    -------
    list of numpy.ndarray
        The six face-vertex arrays.
    """
    hx, hy, hz = half
    corners = np.array(
        [
            [-hx, -hy, -hz],
            [hx, -hy, -hz],
            [hx, hy, -hz],
            [-hx, hy, -hz],
            [-hx, -hy, hz],
            [hx, -hy, hz],
            [hx, hy, hz],
            [-hx, hy, hz],
        ]
    )
    pts = (R @ corners.T).T
    return [pts[idx] for idx in _BOX_FACES]

## Exercise 1 — The inertia tensor and principal axes

Everything begins with $\mathsf I$. For a body aligned with its principal axes the
tensor is diagonal and the moments are the familiar integrals; for a uniform
rectangular box of full side lengths $(a,b,c)$ and mass $m$,
$I_1=\tfrac{m}{12}(b^2+c^2)$ and cyclic. But a body handed to us in an *arbitrary*
orientation has a full (non-diagonal) $\mathsf I$, and diagonalising it
{eq}`eq-inertia` recovers the same three principal moments as eigenvalues. That is
the eigenvalue problem of [§1.5](../01-elementary-mechanics/coupled-oscillators.ipynb) wearing a different hat.

1. Compute the analytic principal moments of a uniform box $(a,b,c)$ with
   $a>b>c$, so $I_1<I_2<I_3$ (the geometry of {numref}`fig-rigid-axes`).
2. Build a *non-aligned* inertia matrix
   $\mathsf I'=\mathsf Q\,\mathrm{diag}(I_1,I_2,I_3)\,\mathsf Q^\top$ for some
   rotation $\mathsf Q$ (`numpy.linalg.qr` on a seeded random matrix),
   diagonalise it with `numpy.linalg.eigvalsh`, and confirm the eigenvalues
   are the analytic moments.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — principal moments match the analytic box result

Diagonalising the arbitrarily-oriented inertia tensor must return exactly the three
analytic box moments: orientation cannot change the eigenvalues.

In [ ]:
validate.close(
    eig,
    np.array(I_box),
    "principal moments match the analytic box result",
    rtol=1e-10,
)

## Exercise 2 — Euler's equations: torque-free integration

With the principal frame in hand, the dynamics are Euler's three equations
{eq}`eq-euler-rigid`. For a generic **asymmetric** top ($I_1\neq I_2\neq I_3$) they
are nonlinear and the components $\omega_i(t)$ trade energy back and forth in a
periodic dance, but two combinations never change: the kinetic energy $T$ and the
magnitude $|\mathbf L|$. We integrate and confirm both.

1. Integrate $\boldsymbol\omega(t)$ for an asymmetric top from a generic
   start with `integrate_body` (a `scipy.integrate.solve_ivp`/`DOP853` wrapper
   on a dense `t_eval`) and plot the three components
   ({numref}`fig-rigid-euler`).
2. Track $T(t)$ and $|\mathbf L(t)|$ (the `T_rot` and `L_body` helpers) and
   confirm they are conserved.

In [ ]:
# (solution hidden on the public site)


### Validation 2 — energy and $|\mathbf L|$ are conserved (torque-free)

The two invariants of free rigid-body motion: $T$ and $|\mathbf L|$ may drift only
at the integrator's tolerance.

In [ ]:
validate.conserved(
    T_series, "rotational energy conserved (torque-free)", rel_drift=1e-6
)
validate.conserved(Lmag_series, "|L| conserved (torque-free)", rel_drift=1e-6)

## Exercise 3 — The polhode (worked)

The two invariants have a gorgeous geometric meaning. In $\boldsymbol\omega$-space,
$T=\text{const}$ is an ellipsoid (semi-axes $\sqrt{2T/I_i}$) and
$|\mathbf L|=\text{const}$ is a second ellipsoid; $\boldsymbol\omega(t)$ must lie on *both*,
so it traces their intersection: the **polhode**. Poinsot's construction: the
tip of $\boldsymbol\omega$ rolls along this closed curve on the energy ellipsoid.
We plot it and verify, point by point, that the trajectory never leaves either
constraint surface.

This is a worked exercise; you build the body-in-space animation in Exercise 6.

**Your task.**

1. Plot the energy ellipsoid and overlay the polhode $\boldsymbol\omega(t)$
   from Exercise 2 ({numref}`fig-rigid-polhode`).
2. Confirm every point of the trajectory sits on both the energy ellipsoid
   ($T=T_0$) and the angular-momentum ellipsoid ($|\mathbf L|=L_0$).

In [ ]:
# (solution hidden on the public site)


### Validation 3 — $\boldsymbol\omega$ stays on the energy ellipsoid and momentum ellipsoid

The defining property of the polhode: every sampled $\boldsymbol\omega(t)$
satisfies both $T(\boldsymbol\omega)=T_0$ and $|\mathbf L(\boldsymbol\omega)|=L_0$:
it lives on the intersection of the two ellipsoids.

In [ ]:
validate.close(
    T_rot(w_gen, I_gen),
    np.full(w_gen.shape[1], T0),
    "ω stays on the energy ellipsoid (T constant)",
    rtol=1e-6,
)
validate.close(
    np.linalg.norm(L_body(w_gen, I_gen), axis=0),
    np.full(w_gen.shape[1], L0),
    "ω stays on the angular-momentum ellipsoid (|L| constant)",
    rtol=1e-6,
)

## Exercise 4 — Symmetric-top free precession

Give the body an axis of symmetry, $I_1=I_2$, and Euler's equations simplify
dramatically: $\omega_3$ is locked constant, and the transverse part
$(\omega_1,\omega_2)$ rotates rigidly at the **free-precession rate** $\Omega$
{eq}`eq-precession`. This is the wobble of a thrown frisbee or a slightly-off spin.
We integrate, confirm $\omega_3$ is constant, and measure $\Omega$ from the phase of
the transverse motion to compare with the prediction.

1. Set $I_1=I_2$, integrate with `integrate_body` (`solve_ivp`/`DOP853`),
   and check $\omega_3$ is constant.
2. Measure the precession rate from the phase of $(\omega_1,\omega_2)$
   (`numpy.unwrap` of `numpy.arctan2`, slope by `numpy.polyfit`) and compare
   to $\Omega=(I_3-I_1)/I_1\cdot\omega_3$ ({numref}`fig-rigid-precession`).

In [ ]:
# (solution hidden on the public site)


### Validation 4 — the free-precession rate is $(I_3-I_1)/I_1\cdot\omega_3$

Two facts about the symmetric top: $\omega_3$ is exactly constant, and the
transverse precession rate matches the closed form {eq}`eq-precession`.

In [ ]:
validate.close(
    Omega_meas, Omega_pred, "free precession rate is (I₃−I₁)/I₁·ω₃", rtol=1e-3
)
validate.close(np.ptp(w_sym[2]), 0.0, "ω₃ is constant for the symmetric top", atol=1e-9)

## Exercise 5 — The tennis-racket theorem (worked animation, centrepiece)

Here is the showpiece. Take a fully asymmetric body ($I_1<I_2<I_3$) and spin it
about each principal axis with a *tiny* transverse nudge. The stability criterion
{eq}`eq-stability` predicts a sharp dichotomy: about the smallest and largest axes
the nudge stays small forever (stable spin, a barely-visible wobble), but about the
**intermediate** axis it explodes: the body flips end over end, again and again.
This is the **tennis-racket / Dzhanibekov theorem**, the reason a flipped phone or a
tumbling wing-nut in orbit periodically reverses. We animate all three at once.

This is the worked animation; you build the body-in-space precession animation in
Exercise 6.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5 — stable about the extreme axes, unstable about the intermediate

The tennis-racket theorem as a number: the perturbation stays tiny for spins about
the smallest and largest moments, but grows to order unity (the body tumbles) for
the intermediate axis: exactly the sign of {eq}`eq-stability`.

In [ ]:
validate.check(
    growth_int > 0.5 and growth_min < 0.05 and growth_max < 0.05,
    "spin is stable about the extreme axes, unstable about the intermediate axis",
    f"growth: min {growth_min:.3f}, mid {growth_int:.3f}, max {growth_max:.3f}",
)

## Exercise 6 — Body orientation in space (student-implemented animation)

Euler's equations live entirely in the **body frame**, but to *watch* a top
precess we need its orientation in the **lab frame**. That is the role of the
rotation matrix $\mathsf R(t)$ integrated in `integrate_body` via
$\dot{\mathsf R}=\mathsf R\,[\boldsymbol\omega]_\times$. For the symmetric top, the
symmetry axis $\hat{\mathbf e}_3(t)=\mathsf R(t)\,\hat{\mathbf z}$ sweeps a cone
about the (space-fixed) angular momentum $\mathbf L$: the visual signature of free
precession. A subtlety worth checking: a numerically integrated $\mathsf R$ can
drift away from being a true rotation, so we confirm it stays orthonormal,
$\mathsf R^\top\mathsf R=\mathsf I$.

This is the **student-implemented** animation. The orientation integration is given;
you build the player and make the precession cone visible.

1. Integrate $\mathsf R(t)$ for the symmetric top with `integrate_body`
   (done for you below) and form the symmetry axis $\hat{\mathbf e}_3(t)$ and
   the space-fixed $\mathbf L$.
2. **Build the animation** of the body axes in space: at minimum the symmetry
   axis tracing its cone around $\mathbf L$ — then `plt.close(fig)` and end
   with `ecp.animate.show(anim)`. There are many valid ways to draw it.
3. Confirm $\mathsf R(t)$ stays a proper rotation,
   $\max|\mathsf R^\top\mathsf R-\mathsf I|\approx0$.

> A ✗ on the final check is about the **orientation integration / orthonormality**,
> not the drawing: any correct animation of the same data is fine. If it fails,
> tighten the integrator tolerances or renormalise $\mathsf R$ (Gram–Schmidt) and
> note why.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6 — the integrated orientation stays a proper rotation

The orientation reconstruction is only trustworthy if $\mathsf R(t)$ remains a
rotation: $\mathsf R^\top\mathsf R=\mathsf I$. With the tight integrator tolerances
it holds to well within $10^{-6}$, so no renormalisation is needed here.

In [ ]:
validate.close(
    ortho_err,
    0.0,
    "the integrated orientation stays a proper rotation (RᵀR=I)",
    atol=1e-6,
)

## Notebook summary

- The **inertia tensor** and its principal axes; torque-free **Euler's equations** integrated
  with $|\mathbf L|$ conserved; the polhode traced on the inertia ellipsoid.
- Symmetric-top free precession ($\omega_3$ constant); the **tennis-racket theorem** (the
  intermediate-axis instability) as the centrepiece; and the body's orientation reconstructed
  in space.

## Outlook

- **The heavy symmetric top.** Add a gravity torque and the free precession of
  Exercise 4 acquires a slow **precession** of the symmetry axis about the vertical
  plus a **nutation** (nodding); the cleanest treatment uses the Euler angles and
  the conserved $L_z$, $L_3$, and $E$ (Goldstein ch. 5).
- **Euler angles and gimbal lock.** The full configuration space of orientations is
  $SO(3)$; the Euler-angle chart has a coordinate singularity (gimbal lock) that
  integrating $\mathsf R$ (or a quaternion) directly, as we did, neatly avoids.
- **The Dzhanibekov effect in orbit.** The intermediate-axis tumbling of Exercise 5
  is the famous "tumbling wing-nut" filmed on the ISS: the tennis-racket theorem in
  microgravity.
- **Chandler wobble.** The Earth is an oblate symmetric top with
  $(I_3-I_1)/I_1\approx1/305$; {eq}`eq-precession` predicts a free-precession period
  of order a year: the observed Chandler wobble of the pole.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()